Random Forest

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # SELECT IMPORTANT FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(250, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # RANDOM FOREST MODEL
    # =====================================================
    model = RandomForestClassifier(

        # More trees
        n_estimators=700,

        # Better generalization
        max_depth=12,

        # Prevent overfitting
        min_samples_split=4,
        min_samples_leaf=2,

        # Better feature sampling
        max_features="sqrt",

        # Handle imbalance
        class_weight="balanced",

        # Enable bootstrap
        bootstrap=True,

        random_state=42,
        n_jobs=-1
    )

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    model.fit(X_train, y_train)

    # =====================================================
    # PREDICT PROBABILITIES
    # =====================================================
    y_prob = model.predict_proba(X_test)[:, 1]

    # =====================================================
    # THRESHOLD
    # =====================================================
    THRESHOLD = 0.35

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC-AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 250)

Predicted Distribution:
[15 14]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.70      0.80        20
           1       0.57      0.89      0.70         9

    accuracy                           0.76        29
   macro avg       0.75      0.79      0.75        29
weighted avg       0.82      0.76      0.77        29

ROC-AUC Score: 0.8389

Confusion Matrix:
[[14  6]
 [ 1  8]]

TRAINING FOR PHASE 2
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 897)
Shape After SelectKBest: (141, 250)

Predicted Distribution:
[20  9]

Actual Distribution:
[20 

Random Forest 5 folds

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # SELECT IMPORTANT FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(250, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # RANDOM FOREST MODEL
        # =================================================
        model = RandomForestClassifier(

            # More trees
            n_estimators=700,

            # Better generalization
            max_depth=12,

            # Prevent overfitting
            min_samples_split=4,
            min_samples_leaf=2,

            # Better feature sampling
            max_features="sqrt",

            # Handle imbalance
            class_weight="balanced",

            # Enable bootstrap
            bootstrap=True,

            random_state=42,
            n_jobs=-1
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 250)

----- Fold 1 -----
Fold ROC-AUC: 0.5556

----- Fold 2 -----
Fold ROC-AUC: 0.9375

----- Fold 3 -----
Fold ROC-AUC: 0.8875

----- Fold 4 -----
Fold ROC-AUC: 0.8937

----- Fold 5 -----
Fold ROC-AUC: 0.8772

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.5556 0.9375 0.8875 0.8938 0.8772]

Mean Fold ROC-AUC:
0.8303

Overall ROC-AUC:
0.8276

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.79      0.83        99
           1       0.59      0.71      0.65        42

    accuracy                           0.77       141
   macro avg       0.73      0.75      0.74       141
weighted avg       0.78      0.77      0.77       1

Random Forest 10 folds

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # SELECT IMPORTANT FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(250, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # RANDOM FOREST MODEL
        # =================================================
        model = RandomForestClassifier(

            # More trees
            n_estimators=700,

            # Better generalization
            max_depth=12,

            # Prevent overfitting
            min_samples_split=4,
            min_samples_leaf=2,

            # Better feature sampling
            max_features="sqrt",

            # Handle imbalance
            class_weight="balanced",

            # Enable bootstrap
            bootstrap=True,

            random_state=42,
            n_jobs=-1
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 250)

----- Fold 1 -----
Fold ROC-AUC: 0.4

----- Fold 2 -----
Fold ROC-AUC: 0.75

----- Fold 3 -----
Fold ROC-AUC: 0.9

----- Fold 4 -----
Fold ROC-AUC: 0.975

----- Fold 5 -----
Fold ROC-AUC: 0.9

----- Fold 6 -----
Fold ROC-AUC: 0.85

----- Fold 7 -----
Fold ROC-AUC: 0.95

----- Fold 8 -----
Fold ROC-AUC: 0.85

----- Fold 9 -----
Fold ROC-AUC: 0.925

----- Fold 10 -----
Fold ROC-AUC: 0.8667

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.4    0.75   0.9    0.975  0.9    0.85   0.95   0.85   0.925  0.8667]

Mean Fold ROC-AUC:
0.8367

Overall ROC-AUC:
0.8307

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.81      0.84   

XGBoost

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP MORE MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(400, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # HANDLE CLASS IMBALANCE
    # =====================================================
    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)

    scale_pos_weight = neg / pos

    print("Scale Pos Weight:", round(scale_pos_weight, 4))

    # =====================================================
    # TUNED MULTIMODAL XGBOOST
    # =====================================================
    model = XGBClassifier(

        # Core
        objective="binary:logistic",
        eval_metric="auc",

        # More boosting
        n_estimators=900,

        # Slightly deeper trees
        max_depth=7,

        # Slower learning
        learning_rate=0.015,

        # Weaker regularization
        reg_alpha=0.3,
        reg_lambda=2,

        # Better multimodal fusion
        subsample=0.9,
        colsample_bytree=0.9,

        # More flexible splits
        min_child_weight=1,
        gamma=0.2,

        # Handle imbalance
        scale_pos_weight=scale_pos_weight,

        # Faster and stable tree method
        tree_method="hist",

        random_state=42,
        n_jobs=-1
    )

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    model.fit(X_train, y_train)

    # =====================================================
    # PREDICT PROBABILITIES
    # =====================================================
    y_prob = model.predict_proba(X_test)[:, 1]

    # =====================================================
    # LOWER THRESHOLD
    # =====================================================
    THRESHOLD = 0.30

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC-AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 400)
Scale Pos Weight: 2.3939

Predicted Distribution:
[17 12]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.70      0.76        20
           1       0.50      0.67      0.57         9

    accuracy                           0.69        29
   macro avg       0.66      0.68      0.66        29
weighted avg       0.72      0.69      0.70        29

ROC-AUC Score: 0.6722

Confusion Matrix:
[[14  6]
 [ 3  6]]

TRAINING FOR PHASE 2
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 897)
Shape After SelectKBest: (141, 400)
Scale Pos Weight: 2.3939

Predicte

XGBoost 5 folds

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(350, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # RANDOM FOREST MODEL
        # =================================================
        model = RandomForestClassifier(

            # More trees
            n_estimators=800,

            # Better multimodal learning
            max_depth=14,

            # Reduce overfitting
            min_samples_split=3,
            min_samples_leaf=1,

            # Better feature selection
            max_features="sqrt",

            # Handle imbalance
            class_weight="balanced",

            # Bootstrap sampling
            bootstrap=True,

            random_state=42,
            n_jobs=-1
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.30

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 350)

----- Fold 1 -----
Fold ROC-AUC: 0.5333

----- Fold 2 -----
Fold ROC-AUC: 0.9187

----- Fold 3 -----
Fold ROC-AUC: 0.9

----- Fold 4 -----
Fold ROC-AUC: 0.9437

----- Fold 5 -----
Fold ROC-AUC: 0.8947

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.5333 0.9188 0.9    0.9438 0.8947]

Mean Fold ROC-AUC:
0.8381

Overall ROC-AUC:
0.8201

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.71      0.80        99
           1       0.55      0.86      0.67        42

    accuracy                           0.75       141
   macro avg       0.74      0.78      0.74       141
weighted avg       0.81      0.75      0.76       141


XGBoost 10 folds

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(350, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # RANDOM FOREST MODEL
        # =================================================
        model = RandomForestClassifier(

            # More trees
            n_estimators=800,

            # Better multimodal learning
            max_depth=14,

            # Reduce overfitting
            min_samples_split=3,
            min_samples_leaf=1,

            # Better feature selection
            max_features="sqrt",

            # Handle imbalance
            class_weight="balanced",

            # Bootstrap sampling
            bootstrap=True,

            random_state=42,
            n_jobs=-1
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.30

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 350)

----- Fold 1 -----
Fold ROC-AUC: 0.36

----- Fold 2 -----
Fold ROC-AUC: 0.725

----- Fold 3 -----
Fold ROC-AUC: 0.9

----- Fold 4 -----
Fold ROC-AUC: 0.975

----- Fold 5 -----
Fold ROC-AUC: 0.95

----- Fold 6 -----
Fold ROC-AUC: 0.875

----- Fold 7 -----
Fold ROC-AUC: 1.0

----- Fold 8 -----
Fold ROC-AUC: 0.925

----- Fold 9 -----
Fold ROC-AUC: 0.85

----- Fold 10 -----
Fold ROC-AUC: 0.8444

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.36   0.725  0.9    0.975  0.95   0.875  1.     0.925  0.85   0.8444]

Mean Fold ROC-AUC:
0.8404

Overall ROC-AUC:
0.8232

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.72      0.80

SVM

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(240, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # FEATURE SCALING
    # =====================================================
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =====================================================
    # BALANCED PHASE-1 SVM
    # =====================================================
    model = SVC(

        # Nonlinear boundary
        kernel="rbf",

        # Balanced flexibility
        C=1.0,

        # Moderate kernel smoothness
        gamma=0.004,

        # Handle imbalance
        class_weight="balanced",

        # Enable probabilities
        probability=True,

        random_state=42
    )

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    model.fit(X_train, y_train)

    # =====================================================
    # PREDICT PROBABILITIES
    # =====================================================
    y_prob = model.predict_proba(X_test)[:, 1]

    # =====================================================
    # THRESHOLD
    # =====================================================
    THRESHOLD = 0.34

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC-AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 240)

Predicted Distribution:
[19 10]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.75      0.77        20
           1       0.50      0.56      0.53         9

    accuracy                           0.69        29
   macro avg       0.64      0.65      0.65        29
weighted avg       0.70      0.69      0.69        29

ROC-AUC Score: 0.7833

Confusion Matrix:
[[15  5]
 [ 4  5]]

TRAINING FOR PHASE 2
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 897)
Shape After SelectKBest: (141, 240)

Predicted Distribution:
[19 10]

Actual Distribution:
[20 

SVM 5 folds

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(240, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # BALANCED PHASE-1 SVM
        # =================================================
        model = SVC(

            # Nonlinear boundary
            kernel="rbf",

            # Balanced flexibility
            C=1.0,

            # Moderate kernel smoothness
            gamma=0.004,

            # Handle imbalance
            class_weight="balanced",

            # Enable probabilities
            probability=True,

            random_state=42
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.34

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 240)

----- Fold 1 -----
Fold ROC-AUC: 0.6

----- Fold 2 -----
Fold ROC-AUC: 0.9125

----- Fold 3 -----
Fold ROC-AUC: 0.7875

----- Fold 4 -----
Fold ROC-AUC: 0.8813

----- Fold 5 -----
Fold ROC-AUC: 0.9035

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.6    0.9125 0.7875 0.8812 0.9035]

Mean Fold ROC-AUC:
0.817

Overall ROC-AUC:
0.8093

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.81      0.82        99
           1       0.59      0.64      0.61        42

    accuracy                           0.76       141
   macro avg       0.71      0.73      0.72       141
weighted avg       0.77      0.76      0.76       141



SVM 10 folds

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(240, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # BALANCED PHASE-1 SVM
        # =================================================
        model = SVC(

            # Nonlinear boundary
            kernel="rbf",

            # Balanced flexibility
            C=1.0,

            # Moderate kernel smoothness
            gamma=0.004,

            # Handle imbalance
            class_weight="balanced",

            # Enable probabilities
            probability=True,

            random_state=42
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.34

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 240)

----- Fold 1 -----
Fold ROC-AUC: 0.46

----- Fold 2 -----
Fold ROC-AUC: 0.725

----- Fold 3 -----
Fold ROC-AUC: 0.95

----- Fold 4 -----
Fold ROC-AUC: 0.9

----- Fold 5 -----
Fold ROC-AUC: 0.9

----- Fold 6 -----
Fold ROC-AUC: 0.75

----- Fold 7 -----
Fold ROC-AUC: 0.925

----- Fold 8 -----
Fold ROC-AUC: 0.875

----- Fold 9 -----
Fold ROC-AUC: 0.925

----- Fold 10 -----
Fold ROC-AUC: 0.9333

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.46   0.725  0.95   0.9    0.9    0.75   0.925  0.875  0.925  0.9333]

Mean Fold ROC-AUC:
0.8343

Overall ROC-AUC:
0.8256

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.82      0.84

Logistic Regression

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # FEATURE SCALING
    # =====================================================
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =====================================================
    # LOGISTIC REGRESSION MODEL
    # =====================================================
    model = LogisticRegression(

        # Better convergence
        solver="saga",

        # Moderate regularization
        C=0.8,

        # Handle imbalance
        class_weight="balanced",

        # More iterations
        max_iter=5000,

        random_state=42
    )

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    model.fit(X_train, y_train)

    # =====================================================
    # PREDICT PROBABILITIES
    # =====================================================
    y_prob = model.predict_proba(X_test)[:, 1]

    # =====================================================
    # THRESHOLD
    # =====================================================
    THRESHOLD = 0.34

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC-AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

Predicted Distribution:
[18 11]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.75      0.79        20
           1       0.55      0.67      0.60         9

    accuracy                           0.72        29
   macro avg       0.69      0.71      0.69        29
weighted avg       0.74      0.72      0.73        29

ROC-AUC Score: 0.7944

Confusion Matrix:
[[15  5]
 [ 3  6]]

TRAINING FOR PHASE 2
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 897)
Shape After SelectKBest: (141, 220)

Predicted Distribution:
[19 10]

Actual Distribution:
[20 

Logistic Regression 5 folds

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # LOGISTIC REGRESSION MODEL
        # =================================================
        model = LogisticRegression(

            # Better convergence
            solver="saga",

            # Moderate regularization
            C=0.8,

            # Handle imbalance
            class_weight="balanced",

            # More iterations
            max_iter=5000,

            random_state=42
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.34

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

----- Fold 1 -----
Fold ROC-AUC: 0.7222

----- Fold 2 -----
Fold ROC-AUC: 0.8

----- Fold 3 -----
Fold ROC-AUC: 0.9125

----- Fold 4 -----
Fold ROC-AUC: 0.8063

----- Fold 5 -----
Fold ROC-AUC: 0.8421

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.7222 0.8    0.9125 0.8062 0.8421]

Mean Fold ROC-AUC:
0.8166

Overall ROC-AUC:
0.828

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.77      0.82        99
           1       0.57      0.74      0.65        42

    accuracy                           0.76       141
   macro avg       0.72      0.75      0.73       141
weighted avg       0.78      0.76      0.77       141



Logistic Regression 10 folds

In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CV
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # LOGISTIC REGRESSION MODEL
        # =================================================
        model = LogisticRegression(

            # Better convergence
            solver="saga",

            # Moderate regularization
            C=0.8,

            # Handle imbalance
            class_weight="balanced",

            # More iterations
            max_iter=5000,

            random_state=42
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(X_train, y_train)

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.34

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

----- Fold 1 -----
Fold ROC-AUC: 0.6

----- Fold 2 -----
Fold ROC-AUC: 0.75

----- Fold 3 -----
Fold ROC-AUC: 0.85

----- Fold 4 -----
Fold ROC-AUC: 0.975

----- Fold 5 -----
Fold ROC-AUC: 1.0

----- Fold 6 -----
Fold ROC-AUC: 0.925

----- Fold 7 -----
Fold ROC-AUC: 0.85

----- Fold 8 -----
Fold ROC-AUC: 0.75

----- Fold 9 -----
Fold ROC-AUC: 0.925

----- Fold 10 -----
Fold ROC-AUC: 0.8222

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.6    0.75   0.85   0.975  1.     0.925  0.85   0.75   0.925  0.8222]

Mean Fold ROC-AUC:
0.8447

Overall ROC-AUC:
0.8297

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.82      0.84 

CatBoost

In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # FEATURE SCALING
    # =====================================================
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =====================================================
    # HANDLE CLASS IMBALANCE
    # =====================================================
    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)

    class_weights = [
        1,
        neg / pos
    ]

    print("Class Weights:", class_weights)

    # =====================================================
    # PHASE-1 OPTIMIZED MULTIMODAL CATBOOST
    # =====================================================
    model = CatBoostClassifier(

        # Core
        loss_function="Logloss",
        eval_metric="AUC",

        # Moderate boosting
        iterations=450,

        # Simpler trees
        depth=4,

        # Slower learning
        learning_rate=0.018,

        # Stronger regularization
        l2_leaf_reg=8,

        # Lower randomness
        random_strength=1,

        # Conservative sampling
        bootstrap_type="Bernoulli",
        subsample=0.75,

        # Handle imbalance
        class_weights=class_weights,

        # Stable growth
        grow_policy="SymmetricTree",

        random_seed=42,

        verbose=0
    )

    # =====================================================
    # TRAIN MODEL
    # =====================================================
    model.fit(
        X_train,
        y_train
    )

    # =====================================================
    # PREDICT PROBABILITIES
    # =====================================================
    y_prob = model.predict_proba(X_test)[:, 1]

    # =====================================================
    # THRESHOLD
    # =====================================================
    THRESHOLD = 0.36

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # =====================================================
    # ROC-AUC
    # =====================================================
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)
Class Weights: [1, np.float64(2.393939393939394)]

Predicted Distribution:
[13 16]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.60      0.73        20
           1       0.50      0.89      0.64         9

    accuracy                           0.69        29
   macro avg       0.71      0.74      0.68        29
weighted avg       0.79      0.69      0.70        29

ROC-AUC Score: 0.7944

Confusion Matrix:
[[12  8]
 [ 1  8]]

TRAINING FOR PHASE 2
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 897)
Shape After SelectKBest: (141, 220)
Class Wei

CatBoost 5 folds

In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CROSS VALIDATION
    # =====================================================
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # HANDLE CLASS IMBALANCE
        # =================================================
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        class_weights = [
            1,
            neg / pos
        ]

        # =================================================
        # PHASE-1 OPTIMIZED MULTIMODAL CATBOOST
        # =================================================
        model = CatBoostClassifier(

            # Core
            loss_function="Logloss",
            eval_metric="AUC",

            # Moderate boosting
            iterations=450,

            # Simpler trees
            depth=4,

            # Slower learning
            learning_rate=0.018,

            # Stronger regularization
            l2_leaf_reg=8,

            # Lower randomness
            random_strength=1,

            # Conservative sampling
            bootstrap_type="Bernoulli",
            subsample=0.75,

            # Handle imbalance
            class_weights=class_weights,

            # Stable growth
            grow_policy="SymmetricTree",

            random_seed=42,

            verbose=0
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(
            X_train,
            y_train
        )

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.36

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

----- Fold 1 -----
Fold ROC-AUC: 0.55

----- Fold 2 -----
Fold ROC-AUC: 0.9062

----- Fold 3 -----
Fold ROC-AUC: 0.8687

----- Fold 4 -----
Fold ROC-AUC: 0.925

----- Fold 5 -----
Fold ROC-AUC: 0.8538

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.55   0.9062 0.8688 0.925  0.8538]

Mean Fold ROC-AUC:
0.8208

Overall ROC-AUC:
0.8112

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.80      0.84        99
           1       0.62      0.79      0.69        42

    accuracy                           0.79       141
   macro avg       0.76      0.79      0.77       141
weighted avg       0.82      0.79      0.80       141


CatBoost 10 folds

In [22]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

# =========================================================
# 1️⃣ LOAD DATASET
# =========================================================
df = pd.read_csv("../data/multimodal_audio_text_features.csv")

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset Shape:", df.shape)

# Normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# =========================================================
# 2️⃣ REMOVE NON-FEATURE COLUMNS
# =========================================================
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# =========================================================
# 3️⃣ TRAINING FUNCTION
# =========================================================
def train_phase_model(df_phase, phase_name):

    print("\n" + "=" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("=" * 60)

    # =====================================================
    # DROP MISSING VALUES
    # =====================================================
    df_phase = df_phase.dropna()

    # =====================================================
    # FEATURES & LABELS
    # =====================================================
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # =====================================================
    # REMOVE LOW VARIANCE FEATURES
    # =====================================================
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # =====================================================
    # KEEP IMPORTANT MULTIMODAL FEATURES
    # =====================================================
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(220, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # =====================================================
    # 5-FOLD STRATIFIED CROSS VALIDATION
    # =====================================================
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # Store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # =====================================================
    # TRAIN EACH FOLD
    # =====================================================
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # =================================================
        # FEATURE SCALING
        # =================================================
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # =================================================
        # HANDLE CLASS IMBALANCE
        # =================================================
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        class_weights = [
            1,
            neg / pos
        ]

        # =================================================
        # PHASE-1 OPTIMIZED MULTIMODAL CATBOOST
        # =================================================
        model = CatBoostClassifier(

            # Core
            loss_function="Logloss",
            eval_metric="AUC",

            # Moderate boosting
            iterations=450,

            # Simpler trees
            depth=4,

            # Slower learning
            learning_rate=0.018,

            # Stronger regularization
            l2_leaf_reg=8,

            # Lower randomness
            random_strength=1,

            # Conservative sampling
            bootstrap_type="Bernoulli",
            subsample=0.75,

            # Handle imbalance
            class_weights=class_weights,

            # Stable growth
            grow_policy="SymmetricTree",

            random_seed=42,

            verbose=0
        )

        # =================================================
        # TRAIN MODEL
        # =================================================
        model.fit(
            X_train,
            y_train
        )

        # =================================================
        # PREDICT
        # =================================================
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.36

        y_pred = (y_prob > THRESHOLD).astype(int)

        # Store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # =================================================
        # FOLD ROC-AUC
        # =================================================
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # =====================================================
    # FINAL RESULTS
    # =====================================================
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "=" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("=" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# =========================================================
# 4️⃣ FILTER DATA PHASE-WISE
# =========================================================
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# =========================================================
# 5️⃣ TRAIN MODELS
# =========================================================
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# =========================================================
# 6️⃣ FINAL SUMMARY
# =========================================================
print("\n" + "=" * 60)
print("FINAL ROC-AUC SCORES")
print("=" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

DATASET INFORMATION
Dataset Shape: (423, 911)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 908

TRAINING FOR PHASE 1
Original Shape: (141, 908)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 898)
Shape After SelectKBest: (141, 220)

----- Fold 1 -----
Fold ROC-AUC: 0.36

----- Fold 2 -----
Fold ROC-AUC: 0.725

----- Fold 3 -----
Fold ROC-AUC: 0.825

----- Fold 4 -----
Fold ROC-AUC: 1.0

----- Fold 5 -----
Fold ROC-AUC: 0.875

----- Fold 6 -----
Fold ROC-AUC: 0.8

----- Fold 7 -----
Fold ROC-AUC: 1.0

----- Fold 8 -----
Fold ROC-AUC: 0.85

----- Fold 9 -----
Fold ROC-AUC: 0.9

----- Fold 10 -----
Fold ROC-AUC: 0.8444

FINAL RESULTS FOR PHASE 1

Fold ROC-AUC Scores:
[0.36   0.725  0.825  1.     0.875  0.8    1.     0.85   0.9    0.8444]

Mean Fold ROC-AUC:
0.8179

Overall ROC-AUC:
0.8076

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.75      0.81   